In [1]:
import psycopg2
import os
import io
from dotenv import load_dotenv
import pandas as pd
import sqlalchemy

In [2]:
load_dotenv(dotenv_path="../../.env")

host=os.getenv("host")
dbname=os.getenv("dbname")
user=os.getenv("user")
password=os.getenv("password")
port=os.getenv("port")

engine = sqlalchemy.create_engine(f'postgresql://{user}:{password}@{host}:{port}/{dbname}')


1) Qual é o total de consultas realizadas por especialidade?

In [4]:
query_1 = """
        SELECT especialidade,
        count(*) AS tot_consulta
        FROM case_2.consultas_silver
        GROUP BY especialidade
        ORDER BY tot_consulta DESC
        """
Q1 = pd.read_sql(query_1, engine)

Q1

,especialidade,tot_consulta
0,ortopedia,81
1,cardiologia,65
2,dermatologia,46
3,ginecologia,36
4,pediatria,34
5,neurologia,32
6,oftalmologia,31
7,psiquiatria,29


2) Qual convênio gerou mais receita total para a clínica?

In [9]:
query_2 = """
        SELECT convenio,
        sum(valor_consulta) AS receita_convenio
        FROM case_2.consultas_silver
        GROUP BY convenio
        ORDER BY receita_convenio DESC
        """
Q2 = pd.read_sql(query_2, engine)

Q2

,convenio,receita_convenio
0,sulamérica,22100.0
1,bradesco saúde,21040.0
2,unimed,19940.0
3,particular,19800.0
4,amil,15490.0


3) Qual médico tem a melhor avaliação média entre os que têm ao menos 10 avaliações válidas?

In [34]:
query_3 = """
    WITH t1 AS (
        SELECT nome_medico,
               avaliacao_paciente::numeric
        FROM case_2.consultas_silver
        WHERE avaliacao_paciente NOT IN ('invalido', 'nao avaliado')
    )
    SELECT nome_medico, 
           round(avg(avaliacao_paciente),2) AS media_avaliacao
    FROM t1
    GROUP BY nome_medico
    HAVING count(*) >= 10
    ORDER BY media_avaliacao DESC
    LIMIT 1
"""
Q3 = pd.read_sql(query_3, engine)

Q3

,nome_medico,media_avaliacao
0,renata souza,3.27


4) Qual a taxa de cancelamento por especialidade mês a mês? Existe alguma especialidade com tendência crescente de cancelamentos?
